## VI velocity maps

In [90]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import os
from astropy.table import Table
import glob
from tqdm import tqdm

import numpy as np

In [2]:
path = '/pscratch/sd/d/dbustos/vel_maps'
no_fiber_path = '/pscratch/sd/d/dbustos/loa_cutouts/cutouts'

In [3]:
shredded = Table.read('/pscratch/sd/d/dbustos/rot_curves/loa_rot_velocity.fits')
shredded[:5]

TARGETID,SGA_ID,TARGET_RA,TARGET_DEC,Z,Z_RR,ZWARN,DELTACHI2,DIST,DIST_R26,PA,C_TO_F_ANGLE,ANGLE_OFF_AXIS,Selection,ZERR_MOD,unique_obs,Velocity,V_err,Z_center,VI,deproj_dist,deproj_r26,manual_VI
int64,int64,float64,float64,float64,float64,int64,float64,float64,float64,float64,float64,float64,int64,float64,float64,float64,float64,float64,int64,float64,float64,int64
6521555517440,1157225,204.22749,32.09493,0.010016011261740405,0.010016011261740405,0,71.96394567680545,0.002217102021260331,0.36627155666222244,65.12960815429688,114.35180431415512,49.22219615985824,1,2.358335538525183e-05,3.0,-6.4195914543075165,13.76801200779403,2.358358388059241e-05,0,0.002881332554681444,0.4760043290496788,0
169678387281926,1430874,257.7495971,56.9324928,0.028921517883227336,0.028921517883227336,0,121857.89663615823,0.003250448915035935,0.31250071154567677,6.557606220245361,206.22810010342067,199.6704938831753,1,2.4024789259983957e-05,3.0,182.8750198451699,11.489101308260244,2.401212431981059e-05,0,0.008096415535868652,0.7783957484224773,0
243323033878571,631434,151.0147419137485,1.6331543609904198,1.3809988331159857,1.3809988331159857,4,0.7242707014083862,0.0057124407591539585,0.9175586924842383,51.69132614135742,32.48938717521324,19.201938966144183,1,5.5595100500533275e-05,8.0,472239.65646196203,24.11477245431036,1.7258340278094598e-05,0,0.006601088916040368,1.0602974753074295,0
1004815400304641,188925,193.6701,28.93758,0.0082724267277799,0.0082724267277799,0,20781.262633197082,0.0009923137447555642,0.026284859154548004,48.5860481262207,146.26021571620655,97.67416758998584,2,2.3542643581428787e-05,31.0,-181.63274387448345,72.81809787647651,1.6648346838273807e-05,0,0.0011151277859353821,0.02953801350383857,0
1005093910478848,1309576,234.30175,43.3014166667,0.019607169071382585,0.019607169071382585,0,21025.242969959974,0.004020441905483598,0.2179634595949638,134.6117401123047,320.7967839323471,186.18504382004244,1,2.380730399661915e-05,8.0,173.70559603525007,12.904304444072334,2.3796769180937358e-05,0,0.005374691904048404,0.2913825069989467,0


In [4]:
file_names = sorted(os.listdir(path), key = len)

## VI Velocity Maps

In [8]:
# images to come back and review later
# remove galaxy
sga_id = []

# remove target ids
target_ids = []

#matching sga id to target id
matched = []

flagged_ids = []

In [73]:
counter = 0

for file in file_names[:0]:

    counter += 1

    print(counter)
    
    img_file = os.path.join(path, file)
    
    img = mpl.image.imread(img_file)

    file_name = os.path.splitext(file)
    
    sga_idx = int(file_name[0].split('_')[1])

    targets = shredded[shredded['SGA_ID'] == sga_idx]['TARGETID']
 
    no_fiber_file = os.path.join(no_fiber_path, 'sga_{}.jpg'.format(sga_idx))
    img_no_fiber = mpl.image.imread(no_fiber_file)

    fig, (ax1, ax) = plt.subplots(1, 2, figsize = (16,18), gridspec_kw={'width_ratios': [1, 2]})

    ax.clear()
    ax.imshow(img)
    ax.set_axis_off()

    ax1.clear()
    ax1.imshow(img_no_fiber)
    ax1.set_axis_off()
    
    fig.canvas.draw()

    plt.show()

    print('SGA ID:', sga_idx)

    # for i, j in enumerate(targets):
    #     print(i, ': ',j)

    user_ipt = input('g to remove galaxy, or enter target id numbers, or s to continue: ').strip()
    
    if user_ipt == 'f':
        flagged_ids += [sga_idx]
        print('flagged', file)

    elif user_ipt == 'g':
        sga_id += [sga_idx]
        print('added {} to remove list'.format(sga_idx))
    
    elif user_ipt == 's':
        print('next galaxy')
        continue

    elif user_ipt:

        remove = user_ipt.split()
        
        target_ids += [targets[int(r)] for r in remove]

        matched +=  ['{}: {}'.format(sga_idx, targets[int(r)]) for r in remove]
            
        print('added target ids to remove list')

print('finished!')

finished!


In [71]:
np.save('/pscratch/sd/d/dbustos/VI/vmap/sga_id',sga_id)
np.save('/pscratch/sd/d/dbustos/VI/vmap/target_id',target_ids)
np.save('/pscratch/sd/d/dbustos/VI/vmap/matched',matched)
np.save('/pscratch/sd/d/dbustos/VI/vmap/flagged',flagged_ids)

## Remove Fibers from table

In [82]:
shredded['rot_vel_filing'] = 0

In [84]:
for i in target_ids:
    targ = int(i)
    shredded['rot_vel_filing'][shredded['TARGETID'] == targ] = 1

In [86]:
for i in sga_id:
    shredded['rot_vel_filing'][shredded['SGA_ID'] == i] = 1

In [88]:
shredded_new = shredded[shredded['rot_vel_filing'] == 0]
shredded_new[:5]

TARGETID,SGA_ID,TARGET_RA,TARGET_DEC,Z,Z_RR,ZWARN,DELTACHI2,DIST,DIST_R26,PA,C_TO_F_ANGLE,ANGLE_OFF_AXIS,Selection,ZERR_MOD,unique_obs,Velocity,V_err,Z_center,VI,deproj_dist,deproj_r26,manual_VI,rot_vel_filing
int64,int64,float64,float64,float64,float64,int64,float64,float64,float64,float64,float64,float64,int64,float64,float64,float64,float64,float64,int64,float64,float64,int64,int64
169678387281926,1430874,257.7495971,56.9324928,0.028921517883227336,0.028921517883227336,0,121857.89663615823,0.003250448915035935,0.31250071154567677,6.557606220245361,206.22810010342067,199.6704938831753,1,2.4024789259983957e-05,3.0,182.8750198451699,11.489101308260244,2.401212431981059e-05,0,0.008096415535868652,0.7783957484224773,0,0
243323033878571,631434,151.0147419137485,1.6331543609904198,1.3809988331159857,1.3809988331159857,4,0.7242707014083862,0.0057124407591539585,0.9175586924842383,51.69132614135742,32.48938717521324,19.201938966144183,1,5.5595100500533275e-05,8.0,472239.65646196203,24.11477245431036,1.7258340278094598e-05,0,0.006601088916040368,1.0602974753074295,0,0
1004815400304641,188925,193.6701,28.93758,0.0082724267277799,0.0082724267277799,0,20781.262633197082,0.0009923137447555642,0.026284859154548004,48.5860481262207,146.26021571620655,97.67416758998584,2,2.3542643581428787e-05,31.0,-181.63274387448345,72.81809787647651,1.6648346838273807e-05,0,0.0011151277859353821,0.02953801350383857,0,0
1005093910478848,1309576,234.30175,43.3014166667,0.019607169071382585,0.019607169071382585,0,21025.242969959974,0.004020441905483598,0.2179634595949638,134.6117401123047,320.7967839323471,186.18504382004244,1,2.380730399661915e-05,8.0,173.70559603525007,12.904304444072334,2.3796769180937358e-05,0,0.005374691904048404,0.2913825069989467,0,0
1005093910478849,1309576,234.303916667,43.2961944444,0.0190429075748926,0.0190429075748926,0,37842.76546475291,0.0023170693385198464,0.12561714881549862,134.6117401123047,204.59833750241611,69.98659739011143,1,2.3794128780331918e-05,8.0,-126.47090634603123,37.70665102866408,2.3796769180937358e-05,0,0.0031473131687750006,0.1706278271946506,0,0


## Rerun Criteria Cut

In [75]:
SGA = Table.read('/global/cfs/cdirs/cosmo/data/sga/2020/SGA-2020.fits', 'ELLIPSE')
SGA[:5]

SGA_ID,SGA_GALAXY,GALAXY,PGC,RA_LEDA,DEC_LEDA,MORPHTYPE,PA_LEDA,D25_LEDA,BA_LEDA,Z_LEDA,SB_D25_LEDA,MAG_LEDA,BYHAND,REF,GROUP_ID,GROUP_NAME,GROUP_MULT,GROUP_PRIMARY,GROUP_RA,GROUP_DEC,GROUP_DIAMETER,BRICKNAME,RA,DEC,D26,D26_REF,PA,BA,RA_MOMENT,DEC_MOMENT,SMA_MOMENT,G_SMA50,R_SMA50,Z_SMA50,SMA_SB22,SMA_SB22.5,SMA_SB23,SMA_SB23.5,SMA_SB24,SMA_SB24.5,SMA_SB25,SMA_SB25.5,SMA_SB26,G_MAG_SB22,R_MAG_SB22,Z_MAG_SB22,G_MAG_SB22.5,R_MAG_SB22.5,Z_MAG_SB22.5,G_MAG_SB23,R_MAG_SB23,Z_MAG_SB23,G_MAG_SB23.5,R_MAG_SB23.5,Z_MAG_SB23.5,G_MAG_SB24,R_MAG_SB24,Z_MAG_SB24,G_MAG_SB24.5,R_MAG_SB24.5,Z_MAG_SB24.5,G_MAG_SB25,R_MAG_SB25,Z_MAG_SB25,G_MAG_SB25.5,R_MAG_SB25.5,Z_MAG_SB25.5,G_MAG_SB26,R_MAG_SB26,Z_MAG_SB26,SMA_SB22_ERR,SMA_SB22.5_ERR,SMA_SB23_ERR,SMA_SB23.5_ERR,SMA_SB24_ERR,SMA_SB24.5_ERR,SMA_SB25_ERR,SMA_SB25.5_ERR,SMA_SB26_ERR,G_MAG_SB22_ERR,R_MAG_SB22_ERR,Z_MAG_SB22_ERR,G_MAG_SB22.5_ERR,R_MAG_SB22.5_ERR,Z_MAG_SB22.5_ERR,G_MAG_SB23_ERR,R_MAG_SB23_ERR,Z_MAG_SB23_ERR,G_MAG_SB23.5_ERR,R_MAG_SB23.5_ERR,Z_MAG_SB23.5_ERR,G_MAG_SB24_ERR,R_MAG_SB24_ERR,Z_MAG_SB24_ERR,G_MAG_SB24.5_ERR,R_MAG_SB24.5_ERR,Z_MAG_SB24.5_ERR,G_MAG_SB25_ERR,R_MAG_SB25_ERR,Z_MAG_SB25_ERR,G_MAG_SB25.5_ERR,R_MAG_SB25.5_ERR,Z_MAG_SB25.5_ERR,G_MAG_SB26_ERR,R_MAG_SB26_ERR,Z_MAG_SB26_ERR,G_COG_PARAMS_MTOT,G_COG_PARAMS_M0,G_COG_PARAMS_ALPHA1,G_COG_PARAMS_ALPHA2,G_COG_PARAMS_CHI2,R_COG_PARAMS_MTOT,R_COG_PARAMS_M0,R_COG_PARAMS_ALPHA1,R_COG_PARAMS_ALPHA2,R_COG_PARAMS_CHI2,Z_COG_PARAMS_MTOT,Z_COG_PARAMS_M0,Z_COG_PARAMS_ALPHA1,Z_COG_PARAMS_ALPHA2,Z_COG_PARAMS_CHI2,ELLIPSEBIT
int64,bytes16,bytes29,int64,float64,float64,bytes21,float32,float32,float32,float32,float32,float32,bool,bytes13,int64,bytes35,int16,bool,float64,float64,float32,bytes8,float64,float64,float32,bytes4,float32,float32,float64,float64,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,int32
2,SGA-2020 2,PGC1283207,1283207,228.3770865,5.4232017,S?,152.2,0.36307806,0.724436,0.03463229,23.40448,16.976,False,LEDA-20181114,0,PGC1283207,1,True,228.3770865,5.4232017,0.36307806,2283p055,228.3770803831908,5.423191398593787,0.49470574,SB26,158.20142,0.545691,228.37700918822188,5.4232652570544015,10.897086,3.3509698,3.1147978,3.240862,5.902337,6.9126143,7.941369,8.997992,10.073601,11.199986,12.391357,13.561038,14.841172,16.966799,16.108246,15.486356,16.879545,16.024958,15.400715,16.818878,15.967034,15.341793,16.776297,15.925804,15.300776,16.746685,15.897334,15.272053,16.725166,15.876816,15.2521105,16.708357,15.862035,15.237181,16.696539,15.851936,15.226998,16.689613,15.844313,15.21976,0.013392451,0.02354,0.021872982,0.01736985,0.024445537,0.039866067,0.05026544,0.08455789,0.122911856,0.005682776,0.0054258136,0.0049038026,0.005588406,0.005323561,0.0047632363,0.00543534,0.005177031,0.0046343105,0.0053025587,0.005040888,0.0045181247,0.005206092,0.0049438984,0.0044374703,0.0051483097,0.0048758644,0.0043834248,0.0051032505,0.0048264163,0.004344248,0.0050705094,0.004792021,0.004319857,0.005054293,0.004765629,0.0043044444,16.65942,0.34037337,0.2978292,3.0239506,0.07928849,15.820566,0.2640441,0.34559453,3.3033552,0.003811298,15.195567,0.29826432,0.3001073,3.2333765,0.011723555,0
3,SGA-2020 3,PGC1310416,1310416,202.54443750000002,6.9345944,Sc,159.26,0.4017908,0.7816278,0.073888786,23.498482,16.85,False,LEDA-20181114,1,PGC1310416,1,True,202.54443750000002,6.9345944,0.4017908,

In [76]:
SGA_dict = {}
for i in range(len(SGA)):
    SGA_dict[SGA['SGA_ID'][i]] = i

In [92]:
center_dist_lim = 0.001
unique_dist_lim = 0.01

In [94]:
# redo criteria cut
#empty column to classify galaxy
shredded_new['Selection'] = 0

#for each unique SGA ID
for i in tqdm(np.unique(shredded_new['SGA_ID'])):

    #identify all the galaxies and targets
    targ_id = shredded_new['SGA_ID'] == i
    
    #makes a table of the targets corresponding to this galaxy
    targ = shredded_new[targ_id]

    manual_VI = targ['manual_VI'][0]

    if manual_VI == 1:
        continue
    
    obs = targ[np.abs(targ['Velocity'] < 1000)]

    sga_id = SGA_dict[i] 

    criteria_a = False
    criteria_b = False


#if the galaxy has more than three observations
    if len(obs) >= 3:

#-----
# a
#-----
        #check to see if there is a center observation
        if np.any(obs['deproj_r26'] <= center_dist_lim):
            # test_counter += 1
        
            # check how many unique points on semimajor axis
            not_center = obs[obs['deproj_r26'] > unique_dist_lim]

            if len(not_center) >= 2 and (np.max(not_center['deproj_r26']) - np.min(not_center['deproj_r26'])) >= unique_dist_lim:
                
                #since there 2 unique points, classify it as viable galaxy
                shredded_new['Selection'][targ_id] = 1
                criteria_a = True

# #----
# # b
# #----
        elif len(obs) >= 10:
            distances = sorted(obs['deproj_r26'])
            counter = 0

            for i in range(len(distances)-1):
                if distances[i+1] - distances[i] > unique_dist_lim:
                    counter += 1
            
            if counter >= 10:
                shredded_new['Selection'][targ_id] = 2
                criteria_b = True

        
#----
# c
#----
          
        elif not (criteria_a or criteria_b):
            
            # identify all points on semimajor axis
            on_major = obs[obs['ANGLE_OFF_AXIS'] == 0]

            # identify all points off major axis
            off_major = obs[obs['ANGLE_OFF_AXIS'] != 0]


            if len(on_major) > 0:
            
                #split targets on semimajor axis onto either side of center galaxy
                if (SGA['PA'][sga_id] < 45) or (SGA['PA'][sga_id] > 135):

                    left_index = on_major['TARGET_DEC'] - SGA['DEC'][sga_id] > 0

                else:
                    left_index = on_major['TARGET_RA'] - SGA['RA'][sga_id] > 0
                
                left = on_major[left_index]
                right = on_major[~left_index]

            if len(left) > 0 and len(right) > 0:

                for j in range(len(left)):
                    
                    # check that there are 2 symmetric observations
                    if np.any(np.abs(right['deproj_r26'] - left['deproj_r26'][j]) < unique_dist_lim):

                    #check if there is a third point
                        if (np.any(np.abs(right['deproj_r26'] - left['deproj_r26'][j]) > unique_dist_lim) 
                            or np.any(np.abs(left['deproj_r26'] - left['deproj_r26'][j]) > unique_dist_lim)
                            or np.any(np.abs(off_major['deproj_r26'] - left['deproj_r26'][j]) > unique_dist_lim)):

                            #viable galaxy
                            shredded_new['Selection'][targ_id] = 3

100%|██████████| 3123/3123 [00:03<00:00, 809.20it/s]


## Save filed fibers

In [96]:
shredded_rot_vel = shredded_new[shredded_new['Selection']!=0]

In [97]:
shredded_rot_vel.write('/pscratch/sd/d/dbustos/rot_curves/shredded_vel_VI.fits',format='fits',overwrite=True)